# Profitable App Profiles for the App Store and Google Play

## Introduction

In this project, we analyze mobile app data from the App Store and Google Play to identify which types of apps are more likely to attract a **large** number of users.

Our company builds free apps and relies on in-app advertisements as the main source of revenue. Therefore, the goal of this analysis is to understand what kinds of apps generate the most user engagement and, consequently, higher ad revenue.

## Opening the Data

In this section, we load the datasets and explore their structure to understand what kind of data we are working with.

In [1]:
from csv import reader

opened_file = open('googleplaystore.csv', encoding='utf-8')
read_file = reader(opened_file)
android = list(read_file)
android_header = android[0]
android = android[1:]

opened_file = open('AppleStore.csv', encoding='utf-8')
read_file = reader(opened_file)
ios = list(read_file)
ios_header = ios[0]
ios = ios[1:]

We'll start by opening and exploring these two data sets.
    To make them easier to explore, we created a function named **explore_data()** that we can repeatedly use to print rows in a readable way.

In [2]:
def explore_data(dataset, start, end, rows_and_columns=False):
    dataset_slice = dataset[start:end]    
    for row in dataset_slice:
        print(row)
        print('\n')

    if rows_and_columns:
        print('Number of rows:', len(dataset))
        print('Number of columns:', len(dataset[0]))

## Data Exploration

In this step, we will explore both datasets to understand their structure, including the number of rows and columns, as well as the available features.

This will help us identify which variables are relevant for analyzing app popularity.

In [3]:
print("ANDROID DATA")
print(android_header)
print('\n')
explore_data(android, 0, 3, True)

ANDROID DATA
['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']


['Photo Editor & Candy Camera & Grid & ScrapBook', 'ART_AND_DESIGN', '4.1', '159', '19M', '10,000+', 'Free', '0', 'Everyone', 'Art & Design', 'January 7, 2018', '1.0.0', '4.0.3 and up']


['Coloring book moana', 'ART_AND_DESIGN', '3.9', '967', '14M', '500,000+', 'Free', '0', 'Everyone', 'Art & Design;Pretend Play', 'January 15, 2018', '2.0.0', '4.0.3 and up']


['U Launcher Lite – FREE Live Cool Themes, Hide Apps', 'ART_AND_DESIGN', '4.7', '87510', '8.7M', '5,000,000+', 'Free', '0', 'Everyone', 'Art & Design', 'August 1, 2018', '1.2.4', '4.0.3 and up']


Number of rows: 10841
Number of columns: 13


In [4]:
print("IOS DATA")
print(ios_header)
print('\n')
explore_data(ios, 0, 3, True)

IOS DATA
['id', 'track_name', 'size_bytes', 'currency', 'price', 'rating_count_tot', 'rating_count_ver', 'user_rating', 'user_rating_ver', 'ver', 'cont_rating', 'prime_genre', 'sup_devices.num', 'ipadSc_urls.num', 'lang.num', 'vpp_lic']


['284882215', 'Facebook', '389879808', 'USD', '0.0', '2974676', '212', '3.5', '3.5', '95.0', '4+', 'Social Networking', '37', '1', '29', '1']


['389801252', 'Instagram', '113954816', 'USD', '0.0', '2161558', '1289', '4.5', '4.0', '10.23', '12+', 'Photo & Video', '37', '0', '29', '1']


['529479190', 'Clash of Clans', '116476928', 'USD', '0.0', '2130805', '579', '4.5', '4.5', '9.24.12', '9+', 'Games', '38', '5', '18', '1']


Number of rows: 7197
Number of columns: 16


## Removing Incorrect Data

While exploring the Google Play dataset, we found a row with incorrect structure reported in the discussion section.

This row is going to be removed to ensure the dataset is consistent and suitable for analysis.

In [5]:
print(android[10472])

['Life Made WI-Fi Touchscreen Photo Frame', '1.9', '19', '3.0M', '1,000+', 'Free', '0', 'Everyone', '', 'February 11, 2018', '1.0.19', '4.0 and up']


---
Here we see that this entry has missing 'Rating' and a column shift happened for next columns.

In [6]:
del android[10472]

In [7]:
print(android[10472]) #checking

['osmino Wi-Fi: free WiFi', 'TOOLS', '4.2', '134203', '4.1M', '10,000,000+', 'Free', '0', 'Everyone', 'Tools', 'August 7, 2018', '6.06.14', '4.4 and up']


The row was removed successfully.

#### App Store Dataset

No obvious incorrect data entries were reported in the discussion section for the App Store dataset.

## Exploring Duplicate Entries

While exploring the Google Play dataset, we noticed that some apps have duplicate entries. 
For example, let's look at the "Instagram" app entries.

In [8]:
for app in android:
    name = app[0]
    if name == "Instagram":
        print(app)
        print("\n")

['Instagram', 'SOCIAL', '4.5', '66577313', 'Varies with device', '1,000,000,000+', 'Free', '0', 'Teen', 'Social', 'July 31, 2018', 'Varies with device', 'Varies with device']


['Instagram', 'SOCIAL', '4.5', '66577446', 'Varies with device', '1,000,000,000+', 'Free', '0', 'Teen', 'Social', 'July 31, 2018', 'Varies with device', 'Varies with device']


['Instagram', 'SOCIAL', '4.5', '66577313', 'Varies with device', '1,000,000,000+', 'Free', '0', 'Teen', 'Social', 'July 31, 2018', 'Varies with device', 'Varies with device']


['Instagram', 'SOCIAL', '4.5', '66509917', 'Varies with device', '1,000,000,000+', 'Free', '0', 'Teen', 'Social', 'July 31, 2018', 'Varies with device', 'Varies with device']




Now, let's quantify the scope of the problem by counting the number of duplicate entries in the Android dataset. We will use a loop to identify apps that have already been encountered.

In [9]:
duplicate_apps = []
unique_apps = []

for app in android:
    name = app[0]
    if name in unique_apps:
        duplicate_apps.append(name)
    else:
        unique_apps.append(name)

print("Number of duplicate apps:", len(duplicate_apps))

Number of duplicate apps: 1181


So, we found 1,181 duplicate entries in the Google Play dataset.

### Strategy for Removing Duplicates

We will not remove the duplicates randomly. 

If we examine the duplicate entries for an app (like Instagram), we can see that the main difference is in the number of reviews. 
The number of reviews is higher when the data was collected more recently.

**Our Criterion:** We will keep the row with the highest number of reviews for each app and remove the other entries. This ensures we are working with the most up-to-date data available in our dataset.

## Creating a Dictionary of Maximum Reviews

To remove duplicates, we will create a dictionary named `reviews_max`. 
- The **key** will be the app name.
- The **value** will be the highest number of reviews for that app.

This dictionary will help us identify which row to keep for each app. We will iterate through the dataset and update the dictionary only if we find a higher number of reviews than what is already stored.

In [10]:
reviews_max = {}

for app in android:
    name = app[0]
    n_reviews = float(app[3])
    
    if name in reviews_max and reviews_max[name] < n_reviews:
        reviews_max[name] = n_reviews
        
    if name not in reviews_max:
        reviews_max[name] = n_reviews

print("Length of reviews_max:", len(reviews_max))

Length of reviews_max: 9659


#### Verification

The length of our `android_clean` dataset is 9,659. Since our original dataset had 10,840 entries (excluding the header) and we identified 1,181 duplicates, our result is correct:

10,840 - 1,181 = 9,659

The data is now clean and ready for further analysis.

## Building the Cleaned Dataset

Now that we have a dictionary containing the maximum number of reviews for each app, we can use it to filter our dataset.

We will create a new list `android_clean` to store our unique rows. We will also use a list called `already_added` to track apps we've already processed, ensuring that we only add one entry per app.

In [11]:
android_clean = []
already_added = []

for app in android:
    name = app[0]
    n_reviews = float(app[3])
    
    if n_reviews == reviews_max[name] and name not in already_added:
        android_clean.append(app)
        already_added.append(name)

print("Length of cleaned dataset:", len(android_clean))    # Verification

Length of cleaned dataset: 9659


### Summary of the Data Cleaning Process

In this section, we performed the following tasks to ensure our data quality:

1. **Identification:** We identified 1,181 duplicate entries in the Google Play dataset.
2. **Strategy:** Instead of removing duplicates randomly, we created a strategy to keep the entry with the highest number of reviews for each app, assuming that more reviews represent more recent data.
3. **Cleaning:** We built a dictionary of maximum reviews and used it to filter the dataset.
4. **Validation:** Our cleaned dataset now contains 9,659 unique entries, confirming the successful removal of all duplicate records.

The dataset is now clean and reliable. We are ready to move forward to the **Exploratory Data Analysis (EDA)** phase, where we will start answering key questions about app profitability and market trends.

## Filtering Non-English Apps

Our goal is to analyze only apps designed for an English-speaking audience. To achieve this, we need to filter out apps with non-English names.

We will use the ASCII standard, where standard English characters (letters, numbers, and basic punctuation) are represented by numbers from 0 to 127. 
We can detect non-English apps by identifying characters with a code greater than 127 using the `ord()` function.

In [12]:
def is_english(string):
    for character in string:
        if ord(character) > 127:
            return False
    return True

# let's test the function
print(is_english('Instagram'))                 # Should return True
print(is_english('爱奇艺PPS -《欢乐颂2》电视剧热播')) # Should return False
print(is_english('Docs To Go™ Free Office Suite'))   # Should return False (due to ™)
print(is_english('Instachat 😜'))               # Should return False (due to emoji)

True
False
False
False


#### Observations
As we can see, our `is_english` function correctly identifies non-English apps and those containing special symbols or emojis. 

**Note on accuracy:** This approach is slightly aggressive. For example, it flags "Docs To Go™" as non-English because the `™` symbol has a code greater than 127. For the purpose of this analysis, we accept this trade-off, as it effectively cleans the majority of non-English entries.

## Refining the Filter Function

To avoid losing valuable data, we will adjust our filter. Instead of strictly removing any app with non-ASCII characters, we will allow up to three such characters. This will keep most English-language apps that contain emojis or symbols (like `™`) while still filtering out apps with entirely non-English names.

In [13]:
def is_english(string):
    non_ascii = 0
    for character in string:
        if ord(character) > 127:
            non_ascii += 1
    
    if non_ascii > 3:
        return False
    return True

# Let's test the updated function
print(is_english('Docs To Go™ Free Office Suite'))   # Should be True
print(is_english('Instachat 😜'))               # Should be True
print(is_english('爱奇艺PPS -《欢乐颂2》电视剧热播')) # Should be False

True
True
False


**Awesome!**

## Filtering Both Datasets

Now, we will apply the updated `is_english` function to both the Google Play (`android_clean`) and App Store (`ios`) datasets. We will iterate through each dataset and retain only those apps that are identified as "English".

In [14]:
android_english = []
ios_english = []

# Filtering Android dataset

for app in android_clean:
    name = app[0]
    if is_english(name):
        android_english.append(app)

# Filtering iOS dataset

for app in ios:
    name = app[1] 
    if is_english(name):
        ios_english.append(app)

print("Android apps remaining:", len(android_english))
print("iOS apps remaining:", len(ios_english))

Android apps remaining: 9614
iOS apps remaining: 6183


In this section, we have completed the filtering of our datasets to focus on apps designed for an English-speaking audience.

Key outcomes:
1. **Heuristic Filter:** We developed a function that identifies non-English apps based on ASCII character encoding.
2. **Data Preservation:** We refined our filter to allow up to three non-ASCII characters. This approach is more robust than a strict filter, as it prevents the accidental removal of English apps containing emojis or symbols (e.g., `™`, `😜`).
3. **Dataset Readiness:** We successfully filtered both the Google Play and App Store datasets.

## Isolating Free Apps

Our business model is based on in-app advertisements, which means we generate revenue only from apps that are free to download. Apps that require payment are not relevant to our analysis.

In the datasets, price is stored as a string. 
- For Google Play, free apps are usually marked as '0'.
- For the App Store, free apps are usually marked as '0.0' or '$0.00'.

We will now filter our `android_english` and `ios_english` lists to retain only the free applications.

In [15]:
android_free = []
ios_free = []

# Filtering Android
for app in android_english:
    price = app[7]
    if price == '0':
        android_free.append(app)

# Filtering iOS: price is at index 4
# We check for '0.0' or '$0' to be safe

for app in ios_english:
    price = app[4]
    if price == '0.0' or price == '$0':
        ios_free.append(app)

# Checking results
print("Free Android apps:", len(android_free))
print("Free iOS apps:", len(ios_free))

Free Android apps: 8864
Free iOS apps: 3222


### Data Cleaning Complete

We have successfully performed the necessary data cleaning tasks:
1. **Removed inaccurate data** (like bad headers or malformed rows).
2. **Removed duplicates** (kept the entry with the highest review count).
3. **Removed non-English apps** (using ASCII encoding logic).
4. **Isolated free apps** (our target segment).
---

## Analyzing App Genres

Our goal is to identify app profiles that succeed in both the App Store and Google Play markets. This helps us maximize our reach and revenue potential.

### Our Validation Strategy
To minimize risk and overhead when developing a new app, we follow a three-step validation strategy:
1. **MVP Development:** Build a minimal Android version of the app and launch it on Google Play.
2. **User Feedback:** If the app receives a positive response from users, we continue development.
3. **Market Expansion:** If the app becomes profitable after six months, we develop an iOS version and launch it on the App Store.

Because our ultimate goal is to launch on both platforms, finding app profiles that perform well in both markets is key to our success. We will begin by analyzing the most common app genres for each market.

In [16]:
# Inspecting the header row of each dataset to identify genre columns
print("Google Play header row:")
print(android_header)

print("\nApp Store header row:")
print(ios_header)

Google Play header row:
['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']

App Store header row:
['id', 'track_name', 'size_bytes', 'currency', 'price', 'rating_count_tot', 'rating_count_ver', 'user_rating', 'user_rating_ver', 'ver', 'cont_rating', 'prime_genre', 'sup_devices.num', 'ipadSc_urls.num', 'lang.num', 'vpp_lic']


### Exploring Data Structure

By inspecting the first rows, we can identify the columns containing genre information:
- **Google Play:** We will use the `Category` and `Genres` columns to understand the app genre distribution.
- **App Store:** We will use the `prime_genre` column.

Now that we know where the data is located, we can build frequency tables to see which genres are most common in each market.

The tedious but essential work of data cleaning is officially behind us. We’ve built a solid, clean, and reliable foundation for our analysis.

Now, it’s time to switch from being "data mechanics" to becoming "data detectives." We are about to dive deep into these datasets to uncover the trends, hidden patterns, and "gold mines" that will guide our product strategy.

Let’s get to the most exciting part — turning this data into actionable business insights!

## Generating Frequency Tables

To understand the app market, we need to compare genres. Dictionaries (frequency tables) are great for counting, but they don't have a built-in order. 

To solve this, we will create two functions:
1. `freq_table()`: This function takes a dataset and an index (column) and returns a dictionary with the frequency of each category expressed as a **percentage**.
2. `display_table()`: This helper function converts our dictionary into a list of tuples, sorts them in descending order, and prints them in a readable format.

In [17]:
def freq_table(dataset, index):
    table = {}
    total = len(dataset)
    
    for row in dataset:
        value = row[index]
        if value in table:
            table[value] += 1
        else:
            table[value] = 1
            
    # Converting counts to percentages
    for key in table:
        table[key] = (table[key] / total) * 100
        
    return table

def display_table(dataset, index):
    table = freq_table(dataset, index)
    table_display = []
    for key in table:
        key_val_as_tuple = (table[key], key)
        table_display.append(key_val_as_tuple)
        
    table_sorted = sorted(table_display, reverse = True)
    for entry in table_sorted:
        print(entry[1], ':', entry[0])

Now, let's use our functions to explore the distribution of app genres in both the App Store and Google Play markets. 

- App Store: `prime_genre` is at index 11.
- Google Play: `Category` is at index 1 and `Genres` is at index 9.

In [18]:
print("App Store: prime_genre\n")
display_table(ios_free, 11)

print("\nGoogle Play: Category\n")
display_table(android_free, 1)

print("\nGoogle Play: Genres\n")
display_table(android_free, 9)

App Store: prime_genre

Games : 58.16263190564867
Entertainment : 7.883302296710118
Photo & Video : 4.9658597144630665
Education : 3.662321539416512
Social Networking : 3.2898820608317814
Shopping : 2.60707635009311
Utilities : 2.5139664804469275
Sports : 2.1415270018621975
Music : 2.0484171322160147
Health & Fitness : 2.0173805090006205
Productivity : 1.7380509000620732
Lifestyle : 1.5828677839851024
News : 1.3345747982619491
Travel : 1.2414649286157666
Finance : 1.1173184357541899
Weather : 0.8690254500310366
Food & Drink : 0.8069522036002483
Reference : 0.5586592178770949
Business : 0.5276225946617008
Book : 0.4345127250155183
Navigation : 0.186219739292365
Medical : 0.186219739292365
Catalogs : 0.12414649286157665

Google Play: Category

FAMILY : 18.907942238267147
GAME : 9.724729241877256
TOOLS : 8.461191335740072
BUSINESS : 4.591606498194946
LIFESTYLE : 3.9034296028880866
PRODUCTIVITY : 3.892148014440433
FINANCE : 3.7003610108303246
MEDICAL : 3.531137184115524
SPORTS : 3.39575812

### App Store Analysis (prime_genre)
**Most Common Genres:**

Games: 58.16%

Entertainment: 7.88%

Photo & Video: 4.97%

**Identified Patterns:** The App Store supply is heavily skewed. The "Games" category accounts for over half of all free apps. A small group of entertainment-focused genres dominates the market, leaving a very small share for practical categories like Education, Utilities, and Productivity.

**General Impression:** The market is primarily designed for entertainment rather than practical utility.

### Google Play Analysis (Category & Genres)
##### Most Common Genres:

**Categories**: Family (18.91%), Game (9.72%), and Tools (8.46%).

**Genres**: Tools (8.45%), Entertainment (6.07%), and Education (5.35%).

**Identified Patterns**: Unlike the App Store, the Google Play market shows a more balanced distribution. While "Family" and "Games" are popular, there is a significant presence of practical categories such as Tools, Business, and Productivity.

**Comparison to App Store**: The App Store appears to be "entertainment-first," whereas Google Play offers a more diverse mix of entertainment and utility. Categories like "Tools" and "Business" maintain a much stronger presence in the Google Play ecosystem than their counterparts do in the App Store.

### Can we recommend an app profile yet?

**No.** Based on frequency tables alone, we cannot recommend a profitable app profile. 

Why? Because frequency tables only describe the **Supply side** of the market (how many apps exist). They tell us nothing about the **Demand side** (how many people use them or how much money they make). 

- Just because there are many "Games" doesn't mean users are downloading them more than "Productivity" apps.
- In fact, the most popular genre might be oversaturated, making it the **worst** choice for a new developer with a limited budget.

To make a data-backed recommendation, we need to analyze user installs (the "popularity" metric) alongside genre.

## Analyzing App Popularity: Measuring Market Demand
### Most Popular Apps by Genre on the App Store
In the previous section, we analyzed **market supply** by calculating the frequency of each app genre. However, a high number of apps in a genre indicates market saturation, not necessarily user engagement. To make a truly data-driven recommendation, we need to understand **User Demand**.

Since the App Store dataset lacks an `Installs` column, we will use `rating_count_tot` (total user ratings) as a **proxy for popularity**. A higher number of ratings strongly correlates with a higher number of installs and, consequently, a larger, more active user base.

### Methodology
To find the genres that actually resonate with users, we will:

1.  **Isolate Genres:** Use our existing frequency table to identify unique genres.
2.  **Calculate Average Popularity:** Use nested loops to iterate through the dataset, summing the total ratings for each genre and dividing by the number of apps in that category.
3.  **Identify Insights:** Compare the "average user engagement" across genres to find categories that are popular with users but potentially less saturated in terms of market supply.

This analysis will allow us to move beyond simple counts and identify the genres where users are truly active and engaged.

In [19]:
genres = freq_table(ios_free, 11)

for genre in genres:
    total = 0
    len_genre = 0
    
    for app in ios_free:
        genre_app = app[11]
        
        if genre_app == genre:
            n_ratings = float(app[5]) # rating_count_tot
            total += n_ratings
            len_genre += 1
            
    avg_n_ratings = total / len_genre
    print(genre, ':', avg_n_ratings)

Social Networking : 71548.34905660378
Photo & Video : 28441.54375
Games : 22788.6696905016
Music : 57326.530303030304
Reference : 74942.11111111111
Health & Fitness : 23298.015384615384
Weather : 52279.892857142855
Utilities : 18684.456790123455
Travel : 28243.8
Shopping : 26919.690476190477
News : 21248.023255813954
Navigation : 86090.33333333333
Lifestyle : 16485.764705882353
Entertainment : 14029.830708661417
Food & Drink : 33333.92307692308
Sports : 23008.898550724636
Book : 39758.5
Finance : 31467.944444444445
Education : 7003.983050847458
Productivity : 21028.410714285714
Business : 7491.117647058823
Catalogs : 4004.0
Medical : 612.0


### Analyzing Popularity: Average Number of User Ratings

We have now calculated the average number of user ratings for each app genre. This acts as a proxy for user engagement and demand, allowing us to distinguish between what developers are building and what users are actually using (demand).

* **Top Genres by Popularity:**
    1. **Navigation** (~86,090 ratings)
    2. **Reference** (~74,942 ratings)
    3. **Social Networking** (~71,548 ratings)

* **Key Observations:**
    * **The "Games" Paradox:** While "Games" are the most frequent genre by a huge margin (constituting ~58% of the App Store supply), they rank surprisingly low in terms of average popularity (13th place with ~22,788 ratings). This confirms that the gaming market is highly saturated and fragmented; the high volume of apps makes it difficult for a new entrant to gain traction.
    * **The "Blue Ocean":** Genres like "Navigation" and "Reference" stand out as clear "Blue Oceans." They have a relatively low number of apps in the App Store but show the highest average user ratings. This suggests that users are actively searching for and relying heavily on these types of practical tools, and there is significantly less competition compared to the entertainment sector.

### App Profile Recommendation

**Recommended App Profile:** An app in the **Navigation** or **Reference** category.

**Rationale:**
1. **Demand vs. Supply:** While "Games" have the highest supply, Navigation and Reference show significantly higher average user engagement. This suggests that users are actively looking for solutions in these areas, but the market is far less saturated than the gaming sector, giving a new app a much better chance to be discovered.
2. **Strategy:** Building a minimal app in this high-demand, lower-competition category allows us to leverage existing, dedicated user interest. Unlike games, which are often "one-off" entertainment, these categories provide ongoing utility, which is more likely to drive long-term engagement and brand loyalty.

---
*Our next step will be to verify these trends in the Google Play dataset to see if we can find a profile that performs well in both markets.*